# `data-conduit` — API Reference

**Auto-generated from source docstrings** — this notebook renders live documentation directly from the codebase,
so it always reflects the current state of the library.

---

### How this notebook is organised

| Section | What it covers |
|---------|----------------|
| **1. Core Workflow Modules** | The modules most users interact with day-to-day: loading data (`io`, `datasource`), combining sources (`multisource`, `virtualarrays`), and querying/segmenting results (`segment`). |
| **2. Synchronisation & Alignment** | Temporal alignment across recording systems: TTL pulse matching (`ttlsync`) and global timebase construction (`globaltimes`). |
| **3. Internals & Utilities** | Lower-level helpers that power the core modules. Useful for building custom workflows, but most users won't need to call these directly (`timestamps`, `harptools`, `utils`, `validators`). |

Click **▶ Parameters** on any item to expand its full parameter documentation.

In [1]:
# ── Shared rendering utilities (run this cell first) ────────────────────────

import html as html_mod
import importlib
import inspect
import re
import textwrap

from IPython.display import HTML, display

# ── Global CSS (injected once) ──────────────────────────────────────────────

_CSS = """
<style>
.dc-ref { font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Helvetica, Arial, sans-serif; color: #24292f; line-height: 1.55; }
.dc-ref h2.dc-section { font-size: 1.5em; font-weight: 600; border-bottom: 2px solid #d0d7de; padding-bottom: 8px; margin: 36px 0 6px 0; }
.dc-ref p.dc-section-desc { color: #57606a; font-size: 0.92em; margin: 0 0 18px 0; }
.dc-ref .dc-mod { margin-bottom: 32px; }
.dc-ref .dc-mod-header { font-size: 1.15em; font-weight: 600; margin: 0 0 4px 0; color: #24292f; }
.dc-ref .dc-mod-path { font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, monospace; font-size: 0.82em; color: #6a737d; margin: 0 0 4px 0; }
.dc-ref .dc-mod-desc { color: #57606a; font-size: 0.9em; margin: 0 0 14px 0; }

/* Individual item — sphinx-style */
.dc-ref .dc-item { border-left: 3px solid #d0d7de; padding: 10px 0 10px 16px; margin-bottom: 6px; }
.dc-ref .dc-item:hover { border-left-color: #0969da; background: #f6f8fa; }
.dc-ref .dc-sig { font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, monospace; font-size: 0.9em; }
.dc-ref .dc-sig .dc-name { font-weight: 700; color: #0550ae; }
.dc-ref .dc-sig .dc-params { color: #57606a; }
.dc-ref .dc-badge { display: inline-block; font-size: 0.62em; font-weight: 700; letter-spacing: 0.6px; text-transform: uppercase;
    padding: 1px 6px; border-radius: 3px; vertical-align: middle; margin-right: 6px; }
.dc-ref .dc-badge-class { background: #6f42c1; color: #fff; }
.dc-ref .dc-badge-func  { background: #0969da; color: #fff; }
.dc-ref .dc-summary { color: #24292f; font-size: 0.88em; margin: 4px 0 0 0; }

/* Parameter definition list */
.dc-ref .dc-details summary { cursor: pointer; font-size: 0.82em; font-weight: 600; color: #0969da; margin-top: 6px; }
.dc-ref .dc-details summary:hover { text-decoration: underline; }
.dc-ref .dc-dl { margin: 6px 0 0 0; padding: 0; }
.dc-ref .dc-dl dt { font-family: "SFMono-Regular", Consolas, "Liberation Mono", Menlo, monospace; font-size: 0.85em; font-weight: 600; color: #24292f; margin-top: 6px; }
.dc-ref .dc-dl dt .dc-ptype { font-weight: 400; color: #6a737d; margin-left: 4px; }
.dc-ref .dc-dl dd { margin: 1px 0 0 16px; font-size: 0.84em; color: #57606a; }
.dc-ref .dc-returns { font-size: 0.82em; color: #656d76; margin-top: 6px; }
.dc-ref .dc-returns strong { color: #24292f; }

/* Divider between modules */
.dc-ref hr.dc-sep { border: none; border-top: 1px solid #eaeef2; margin: 20px 0; }
</style>
"""


# ── Docstring parser ────────────────────────────────────────────────────────

def _parse_numpy_docstring(doc):
    """Parse a NumPy-style docstring into structured components."""
    result = {"summary": "", "params": [], "returns": "", "returns_type": "", "attributes": [], "raises": []}
    if not doc:
        return result

    lines = doc.strip().splitlines()
    summary_lines, i = [], 0

    # Summary = everything before the first section underline
    while i < len(lines):
        if i + 1 < len(lines) and set(lines[i + 1].strip()) <= {"-"} and lines[i + 1].strip():
            break
        summary_lines.append(lines[i].strip())
        i += 1
    result["summary"] = " ".join(ln for ln in summary_lines if ln).strip()

    # Collect named sections (Header\n------\ncontent...)
    sections = {}
    while i < len(lines):
        header = lines[i].strip()
        if i + 1 < len(lines) and set(lines[i + 1].strip()) <= {"-"} and lines[i + 1].strip():
            sec = []
            i += 2
            while i < len(lines):
                if i + 1 < len(lines) and set(lines[i + 1].strip()) <= {"-"} and lines[i + 1].strip():
                    break
                sec.append(lines[i])
                i += 1
            sections[header.lower()] = sec
        else:
            i += 1

    def _dedent_block(block):
        """Remove common leading whitespace from a block of lines."""
        return textwrap.dedent("\n".join(block)).splitlines()

    def _parse_entries(raw_block):
        """Parse 'name : type\\n    description' entries from a dedented block."""
        block = _dedent_block(raw_block)
        entries, j = [], 0
        while j < len(block):
            line = block[j]
            # Match:  name : type   or   **kwargs : type   or  *args : type
            m = re.match(r"^(\*{0,2}\w[\w\s,*]*?)\s*:\s*(.+)$", line)
            if m:
                pname = m.group(1).strip()
                ptype = m.group(2).strip()
                desc_parts = []
                j += 1
                # Collect indented description lines
                while j < len(block):
                    ln = block[j]
                    if ln.strip() == "":
                        j += 1
                        continue
                    # If line starts with whitespace, it's a continuation
                    if ln[0] in (" ", "\t"):
                        desc_parts.append(ln.strip())
                        j += 1
                    else:
                        break
                entries.append((pname, ptype, " ".join(desc_parts)))
            else:
                j += 1
        return entries

    for key, target in [("parameters", "params"), ("attributes", "attributes"), ("raises", "raises")]:
        if key in sections:
            result[target] = _parse_entries(sections[key])

    if "returns" in sections:
        ret = _dedent_block(sections["returns"])
        type_line = ret[0].strip() if ret else ""
        desc_lines = [l.strip() for l in ret[1:] if l.strip()]
        if type_line and not re.match(r"^\w[\w\s,*]*?\s*:", type_line):
            result["returns_type"] = type_line
            result["returns"] = " ".join(desc_lines)
        else:
            all_lines = [l.strip() for l in ret if l.strip()]
            result["returns"] = " ".join(all_lines)

    return result


def _get_best_docstring(obj):
    """Get the most complete docstring for an object.
    
    For classes: prefer the class docstring if it has a Parameters section,
    otherwise fall back to __init__'s docstring, then merge attributes from
    the class docstring if available.
    """
    if not inspect.isclass(obj):
        return inspect.getdoc(obj) or ""

    class_doc = inspect.getdoc(obj) or ""
    init_doc = inspect.getdoc(obj.__init__) or ""

    # Check which docstring has Parameters
    class_has_params = bool(re.search(r"^Parameters\s*\n\s*-{3,}", class_doc, re.MULTILINE))
    init_has_params = bool(re.search(r"^Parameters\s*\n\s*-{3,}", init_doc, re.MULTILINE))

    if class_has_params:
        return class_doc
    elif init_has_params:
        # Use __init__ doc but steal Attributes section from class doc if present
        class_has_attrs = bool(re.search(r"^Attributes\s*\n\s*-{3,}", class_doc, re.MULTILINE))
        if class_has_attrs and not re.search(r"^Attributes\s*\n\s*-{3,}", init_doc, re.MULTILINE):
            # Extract Attributes section from class doc
            attrs_match = re.search(r"(Attributes\s*\n\s*-{3,}.*?)(?=\n\w+\s*\n-{3,}|\Z)", class_doc, re.DOTALL | re.MULTILINE)
            if attrs_match:
                return init_doc + "\n\n" + attrs_match.group(1)
        return init_doc
    elif class_doc:
        return class_doc
    else:
        return init_doc


# ── HTML renderers ──────────────────────────────────────────────────────────

def _dl(entries, heading):
    """Render parameter/attribute entries as a definition list inside <details>."""
    if not entries:
        return ""
    items = []
    for pn, pt, pd in entries:
        items.append(
            f"<dt>{html_mod.escape(pn)} <span class='dc-ptype'>: {html_mod.escape(pt)}</span></dt>"
            f"<dd>{html_mod.escape(pd)}</dd>"
        )
    dl_body = "".join(items)
    return (
        f"<details class='dc-details'>"
        f"<summary>{heading}</summary>"
        f"<dl class='dc-dl'>{dl_body}</dl>"
        f"</details>"
    )


def _returns_html(parsed):
    rtype = parsed.get("returns_type", "")
    rdesc = parsed.get("returns", "")
    if not rtype and not rdesc:
        return ""
    parts = []
    if rtype:
        parts.append(f"<code>{html_mod.escape(rtype)}</code>")
    if rdesc:
        parts.append(html_mod.escape(rdesc))
    return "<div class='dc-returns'><strong>Returns</strong>&ensp;" + " &mdash; ".join(parts) + "</div>"


def _render_item(name, obj):
    is_class = inspect.isclass(obj)
    badge_cls = "dc-badge-class" if is_class else "dc-badge-func"
    badge_label = "class" if is_class else "func"

    try:
        sig = str(inspect.signature(obj))
    except (ValueError, TypeError):
        sig = "(...)"

    doc = _get_best_docstring(obj)
    parsed = _parse_numpy_docstring(doc)
    summary = html_mod.escape(parsed["summary"]) or "<em>No description available.</em>"

    details = _dl(parsed["params"], "Parameters")
    details += _dl(parsed["attributes"], "Attributes")
    details += _dl(parsed["raises"], "Raises")
    details += _returns_html(parsed)

    return (
        f"<div class='dc-item'>"
        f"<span class='dc-badge {badge_cls}'>{badge_label}</span>"
        f"<span class='dc-sig'><span class='dc-name'>{html_mod.escape(name)}</span>"
        f"<span class='dc-params'>{html_mod.escape(sig)}</span></span>"
        f"<p class='dc-summary'>{summary}</p>"
        f"{details}"
        f"</div>"
    )


def render_module(module, display_name, import_path, description=""):
    items = []
    for name in sorted(module.__all__):
        obj = getattr(module, name, None)
        if obj is not None:
            items.append(_render_item(name, obj))

    desc_html = f"<p class='dc-mod-desc'>{description}</p>" if description else ""
    joined = "\n".join(items)
    return (
        f"<div class='dc-mod'>"
        f"<p class='dc-mod-header'>{display_name}</p>"
        f"<p class='dc-mod-path'>import {html_mod.escape(import_path)}</p>"
        f"{desc_html}"
        f"{joined}"
        f"</div>"
    )


def render_section(specs, section_title="", section_desc=""):
    parts = []
    if section_title:
        parts.append(f"<h2 class='dc-section'>{section_title}</h2>")
        if section_desc:
            parts.append(f"<p class='dc-section-desc'>{section_desc}</p>")

    for i, (display_name, import_path, desc) in enumerate(specs):
        try:
            mod = importlib.import_module(import_path)
            parts.append(render_module(mod, display_name, import_path, desc))
        except Exception as e:
            parts.append(
                f"<div class='dc-mod'>"
                f"<p class='dc-mod-header'>{html_mod.escape(display_name)}</p>"
                f"<p class='dc-mod-path'>import {html_mod.escape(import_path)}</p>"
                f"<p style='color:#cf222e;font-style:italic;font-size:0.88em;'>"
                f"Skipped &mdash; import failed: {html_mod.escape(str(e))}</p></div>"
            )
        if i < len(specs) - 1:
            parts.append("<hr class='dc-sep'>")

    joined = "\n".join(parts)
    return f"<div class='dc-ref'>{joined}</div>"

# Inject CSS once
display(HTML(_CSS))
print("\u2713 Rendering utilities loaded. Run the cells below to generate the reference.")

✓ Rendering utilities loaded. Run the cells below to generate the reference.


---

## 1. Core Workflow Modules

These are the primary modules for day-to-day use. Together they cover the standard
`data-conduit` pipeline:

1. **Discover & read** files from a directory tree (`io`)
2. **Wrap** the resulting data dictionaries with device-aware metadata (`datasource`)
3. **Combine** multiple sources into unified, queryable arrays (`multisource` + `virtualarrays`)
4. **Segment** continuous recordings around events of interest (`segment`)

In [2]:
core_modules = [
    (
        "IO &nbsp; <code>data_conduit.io</code>",
        "data_conduit.io",
        "The main entry point for loading data. <code>collect_dfs</code> walks a directory tree and "
        "returns a nested dictionary of DataFrames, with file readers dispatched by extension. "
        "The reader registry (<code>add_reader</code> / <code>get_reader</code>) lets you plug in "
        "custom formats alongside the built-in CSV, JSON, JSONL, and YAML readers.",
    ),
    (
        "DataSource &nbsp; <code>data_conduit.datasource</code>",
        "data_conduit.datasource",
        "Convenience wrappers around <code>collect_dfs</code> that preconfigure readers, level selectors, "
        "and named DataArray extraction for common experimental data types. "
        "<code>DataSource</code> is the base class; <code>Device</code> handles HARP .bin files; "
        "<code>FileTypeData</code> handles flat file formats. Presets like <code>SoundCard</code>, "
        "<code>ExperimentEvents</code>, and <code>RotationData</code> provide one-line loading for "
        "standard Bonsai workflow outputs.",
    ),
    (
        "MultiSource &nbsp; <code>data_conduit.multisource</code>",
        "data_conduit.multisource",
        "Orchestration layer for combining data from multiple devices or files into "
        "unified xarray structures using virtual coordinate maps. "
        "<code>MultiDevice</code> extends this with HARP device resolution; "
        "<code>Nosepoke</code> is a preset for the 6-device &times; 3-local-ID nosepoke peripheral array.",
    ),
    (
        "Virtual Arrays &nbsp; <code>data_conduit.virtualarrays</code>",
        "data_conduit.virtualarrays",
        "Construct and query n-dimensional xarray DataArrays where <em>virtual coordinates</em> "
        "(e.g. device, register, channel) map onto a single <em>global coordinate</em> axis. "
        "<code>ulookup</code> provides flexible selection by any combination of virtual coordinates, "
        "and the <code>.ulookup()</code> xarray accessor makes this available directly on DataArrays.",
    ),
    (
        "Segmentation &nbsp; <code>data_conduit.segment</code>",
        "data_conduit.segment",
        "Lightweight primitives for slicing time-indexed data. "
        "<code>segment_boolean_series</code> performs run-length encoding on state signals; "
        "<code>slice_event_windows</code> and <code>slice_dataarray_windows</code> cut "
        "DataFrames or DataArrays into event-centred windows.",
    ),
]

display(HTML(render_section(
    core_modules,
    section_title="",
)))

---

## 2. Synchronisation & Alignment

Experimental recordings often come from systems running on independent clocks
(e.g. Bonsai/HARP vs. Neuropixels). These modules align timebases using shared
TTL pulse trains and provide utilities for mapping any stream onto a common
global clock.

In [3]:
sync_modules = [
    (
        "TTL Sync &nbsp; <code>data_conduit.ttlsync</code>",
        "data_conduit.ttlsync",
        "End-to-end TTL synchronisation pipeline: extract pulse segments from raw waveforms, "
        "align pulse tables across clocks, fit a linear timebase model, and apply the conversion. "
        "<code>TTLSyncModel</code> encapsulates the fitted slope/intercept/R\u00b2; "
        "<code>get_npx_to_bonsai_time_conversion</code> is a semantic shortcut for the "
        "common Neuropixels \u2194 Bonsai alignment. Includes visualisation helpers for "
        "inspecting pulse alignment and conversion error.",
    ),
    (
        "Global Times &nbsp; <code>data_conduit.globaltimes</code>",
        "data_conduit.globaltimes",
        "Create a canonical, evenly-spaced global time vector and map heterogeneous stream "
        "timestamps onto it. <code>index_map_util</code> supports nearest, before, after, "
        "and exact matching strategies.",
    ),
]

display(HTML(render_section(
    sync_modules,
    section_title="",
)))

---

## 3. Internals & Utilities

These modules provide the lower-level building blocks that the core modules rely on.
Most users will not need to call them directly, but they are documented here for
completeness and for anyone building custom workflows or extending `data-conduit`.

- **Timestamps** — extract timestamp vectors from DataFrames or nested dicts, used internally by `DataSource` and `MultiSource`.
- **HarpTools** — reader construction and register lookup for HARP `.bin` files, used internally by `Device`.
- **Utils** — selector callables (`starts_with`, `ends_with`, `contains`) for level filtering, plus internal helpers for directory traversal and nested-dict manipulation.
- **Validators** — shared input validation for `dfs_dict` / `base_path` arguments.

In [4]:
internal_modules = [
    (
        "Timestamps &nbsp; <code>data_conduit.timestamps</code>",
        "data_conduit.timestamps",
        "Collect timestamp vectors from individual DataFrames, flat dictionaries, or "
        "arbitrarily nested data structures. Supports flexible lookup by column name, "
        "index, or auto-detection.",
    ),
    (
        "HarpTools &nbsp; <code>data_conduit.harptools</code>",
        "data_conduit.harptools",
        "Extends the <code>harp-python</code> library with device reader construction from "
        "YAML schemas, register address lookup, and a <code>read_harp_bin</code> reader "
        "that plugs into the IO registry.",
    ),
    (
        "Utils &nbsp; <code>data_conduit.utils</code>",
        "data_conduit.utils",
        "Helper callables for level selectors (<code>starts_with</code>, <code>ends_with</code>, "
        "<code>contains</code>) and internal directory-traversal / nested-dict utilities. "
        "The selector helpers are the most commonly used exports.",
    ),
    (
        "Validators &nbsp; <code>data_conduit.validators</code>",
        "data_conduit.validators",
        "Shared validation logic used by <code>DataSource</code> and <code>MultiSource</code> "
        "to resolve whether to build a new <code>dfs_dict</code> from a directory or reuse "
        "a pre-built one.",
    ),
]

display(HTML(render_section(
    internal_modules,
    section_title="",
)))